# Enum — Nível Básico

**Objetivo:** substituir strings e números "mágicos" por constantes nomeadas, sabendo acessar, converter, iterar,
comparar e despachar com `match/case`.

Teoria completa em [`enum.md`](enum.md). Roteiro:

1. O problema das strings mágicas
2. Primeiro Enum: `name` e `value`
3. Convertendo: por valor `Status(...)` e por nome `Status[...]`
4. Iteração, tamanho e pertinência
5. Comparação: `is`, `==` e a diferença para o valor bruto
6. `auto()`
7. Aliases e `@unique`
8. Despacho com `match/case`
9. Valores que não são números
10. Resumo

## 1. O problema das strings mágicas

In [1]:
def ship(order: dict) -> str:
    if order["status"] == "paid":
        return "enviando"
    if order["status"] == "shiped":          # typo: nunca vai casar, e nada avisa
        return "já enviado"
    return "aguardando pagamento"

print(f'ship({{"status": "paid"}}): {ship({"status": "paid"})}')
print(f'ship({{"status": "shipped"}}): {ship({"status": "shipped"})}')         # deveria ser "já enviado"
print(f'ship({{"status": "Paid"}}): {ship({"status": "Paid"})}')            # maiúscula: cai no default

ship({"status": "paid"}): enviando
ship({"status": "shipped"}): aguardando pagamento
ship({"status": "Paid"}): aguardando pagamento


Três bugs silenciosos em seis linhas. Nenhum dá erro; todos produzem o resultado errado.

## 2. Primeiro Enum: `name` e `value`

In [2]:
from enum import Enum

class OrderStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    SHIPPED = "shipped"
    CANCELLED = "cancelled"

s = OrderStatus.PAID
print(f"s: {s}")                 # representação legível
print(f"repr(s): {repr(s)}")           # <Classe.NOME: valor>
print(f"s.name: {s.name}")
print(f"s.value: {s.value}")
print(f"type(s): {type(s)}")
print(f"isinstance(s, OrderStatus): {isinstance(s, OrderStatus)}")

s: OrderStatus.PAID
repr(s): <OrderStatus.PAID: 'paid'>
s.name: PAID
s.value: paid
type(s): <enum 'OrderStatus'>
isinstance(s, OrderStatus): True


Com o Enum, o typo vira **erro imediato** (e o editor/type checker avisa antes mesmo de rodar):

In [3]:
try:
    OrderStatus.SHIPED
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: type object 'OrderStatus' has no attribute 'SHIPED'


## 3. Convertendo: por valor e por nome

Dados de fora (JSON, banco, formulário) chegam como valor bruto. Converta **na entrada**:

| sintaxe | busca por | uso |
|---|---|---|
| `OrderStatus("paid")` | **valor** | dados persistidos / JSON / API |
| `OrderStatus["PAID"]` | **nome** | config, argumentos de CLI |

In [4]:
print(f'OrderStatus("paid"): {OrderStatus("paid")}')
print(f'OrderStatus["SHIPPED"]: {OrderStatus["SHIPPED"]}')

try:
    OrderStatus("PAID")                 # valor é case-sensitive
except ValueError as e:
    print(f"ValueError: {e}")
try:
    OrderStatus["paid"]                 # nome também
except KeyError as e:
    print(f"KeyError: {e}")

OrderStatus("paid"): OrderStatus.PAID
OrderStatus["SHIPPED"]: OrderStatus.SHIPPED
ValueError: 'PAID' is not a valid OrderStatus
KeyError: 'paid'


Membros são **singletons**: toda conversão devolve exatamente o mesmo objeto.

In [5]:
print(f'OrderStatus("paid") is OrderStatus.PAID is OrderStatus["PAID"]: {OrderStatus("paid") is OrderStatus.PAID is OrderStatus["PAID"]}')

OrderStatus("paid") is OrderStatus.PAID is OrderStatus["PAID"]: True


## 4. Iteração, tamanho e pertinência

In [6]:
print(f"list(OrderStatus): {list(OrderStatus)}")
print(f"len(OrderStatus): {len(OrderStatus)}")
print(f"[s.value for s in OrderStatus]: {[s.value for s in OrderStatus]}")
print(f"{{s.name: s.value for s in OrderStatus}}: {({s.name: s.value for s in OrderStatus})}")

print(f"OrderStatus.PAID in OrderStatus: {OrderStatus.PAID in OrderStatus}")
print(f'"paid" in OrderStatus: {"paid" in OrderStatus}  <- por valor, Python 3.12+')
print(f'"refunded" in OrderStatus: {"refunded" in OrderStatus}')

list(OrderStatus): [<OrderStatus.PENDING: 'pending'>, <OrderStatus.PAID: 'paid'>, <OrderStatus.SHIPPED: 'shipped'>, <OrderStatus.CANCELLED: 'cancelled'>]
len(OrderStatus): 4
[s.value for s in OrderStatus]: ['pending', 'paid', 'shipped', 'cancelled']
{s.name: s.value for s in OrderStatus}: {'PENDING': 'pending', 'PAID': 'paid', 'SHIPPED': 'shipped', 'CANCELLED': 'cancelled'}
OrderStatus.PAID in OrderStatus: True
"paid" in OrderStatus: True  <- por valor, Python 3.12+
"refunded" in OrderStatus: False


Isso é ótimo para montar **opções de UI**, validar entradas e documentar o que existe:

In [7]:
def parse_status(raw: str) -> OrderStatus:
    if raw not in OrderStatus:
        options = ", ".join(s.value for s in OrderStatus)
        raise ValueError(f"status {raw!r} inválido. Opções: {options}")
    return OrderStatus(raw)

print(f'parse_status("shipped"): {parse_status("shipped")}')
try:
    parse_status("shiped")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

parse_status("shipped"): OrderStatus.SHIPPED
ValueError: status 'shiped' inválido. Opções: pending, paid, shipped, cancelled


## 5. Comparação: `is`, `==` e o valor bruto

* Como membros são singletons, a comparação idiomática é **`is`** (identidade).
* `==` também funciona entre membros.
* Com `Enum` puro, **membro ≠ valor bruto**: `OrderStatus.PAID == "paid"` é `False`. Isso é proposital — força a conversão na borda.
  (No intermediário veremos `StrEnum`, que muda esse comportamento.)

In [8]:
s = OrderStatus("paid")
print(f"s is OrderStatus.PAID: {s is OrderStatus.PAID}")
print(f"s == OrderStatus.PAID: {s == OrderStatus.PAID}")
print(f's == "paid": {s == "paid"}  <- Enum puro não se compara com o valor bruto')
print(f's.value == "paid": {s.value == "paid"}')

s is OrderStatus.PAID: True
s == OrderStatus.PAID: True
s == "paid": False  <- Enum puro não se compara com o valor bruto
s.value == "paid": True


Enums são **hashable**: servem como chave de dicionário — o jeito clássico de trocar cadeias de `if/elif` por uma tabela.

In [9]:
LABELS_PT = {
    OrderStatus.PENDING: "Aguardando pagamento",
    OrderStatus.PAID: "Pago",
    OrderStatus.SHIPPED: "Enviado",
    OrderStatus.CANCELLED: "Cancelado",
}
print(f'LABELS_PT[OrderStatus("shipped")]: {LABELS_PT[OrderStatus("shipped")]}')

LABELS_PT[OrderStatus("shipped")]: Enviado


## 6. `auto()`

Quando o valor não importa (só a identidade do membro), `auto()` gera os valores. Em `Enum` puro gera inteiros 1, 2, 3...

In [10]:
from enum import auto

class Priority(Enum):
    LOW = auto()
    MEDIUM = auto()
    HIGH = auto()
    CRITICAL = auto()

print(f"[(p.name, p.value) for p in Priority]: {[(p.name, p.value) for p in Priority]}")

[(p.name, p.value) for p in Priority]: [('LOW', 1), ('MEDIUM', 2), ('HIGH', 3), ('CRITICAL', 4)]


> ⚠️ Se esses números forem **gravados** em banco/arquivo, inserir um membro no meio muda os valores dos seguintes e corrompe
> os dados antigos. Para valores persistidos, escreva-os explicitamente (ou use `StrEnum` + `auto()`, que gera o nome — intermediário).

In [11]:
# O problema: alguém insere URGENT entre HIGH e CRITICAL
class PriorityV2(Enum):
    LOW = auto()
    MEDIUM = auto()
    HIGH = auto()
    URGENT = auto()      # novo
    CRITICAL = auto()

saved_in_db = Priority.CRITICAL.value          # gravado como 4 na versão antiga
print(f"Priority.CRITICAL.value (gravado): {saved_in_db}")
print(f"PriorityV2({saved_in_db}) (lido depois do deploy): {PriorityV2(saved_in_db)}  <- virou URGENT!")

Priority.CRITICAL.value (gravado): 4
PriorityV2(4) (lido depois do deploy): PriorityV2.URGENT  <- virou URGENT!


**Correção:** valores persistidos são **explícitos e estáveis** — inserir um membro novo não mexe nos antigos.

In [12]:
class Priority(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    URGENT = "urgent"        # inserido depois, sem afetar ninguém
    CRITICAL = "critical"

print(f'Priority("critical"): {Priority("critical")}')
print(f"[(p.name, p.value) for p in Priority]: {[(p.name, p.value) for p in Priority]}")

Priority("critical"): Priority.CRITICAL
[(p.name, p.value) for p in Priority]: [('LOW', 'low'), ('MEDIUM', 'medium'), ('HIGH', 'high'), ('URGENT', 'urgent'), ('CRITICAL', 'critical')]


## 7. Aliases e `@unique`

Dois nomes com o **mesmo valor** não criam dois membros: o segundo vira **alias** do primeiro e não aparece na iteração.
Às vezes é intencional (nome antigo mantido por compatibilidade), às vezes é um bug de copiar-e-colar.

In [13]:
class Size(Enum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3
    M = 2            # alias de MEDIUM

print(f"list(Size): {list(Size)}")
print(f"Size.M: {Size.M}")
print(f"Size.M is Size.MEDIUM: {Size.M is Size.MEDIUM}")
print(f"list(Size.__members__): {list(Size.__members__)}")     # __members__ inclui aliases

list(Size): [<Size.SMALL: 1>, <Size.MEDIUM: 2>, <Size.LARGE: 3>]
Size.M: Size.MEDIUM
Size.M is Size.MEDIUM: True
list(Size.__members__): ['SMALL', 'MEDIUM', 'LARGE', 'M']


In [14]:
from enum import unique

try:
    @unique
    class Plan(Enum):
        FREE = "free"
        PRO = "pro"
        TEAM = "pro"          # copiar-e-colar
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: duplicate values found in <enum 'Plan'>: TEAM -> PRO


**Correção:** cada membro com o próprio valor. Com `@unique`, a classe só é criada quando não há alias acidental.

In [15]:
@unique
class Plan(Enum):
    FREE = "free"
    PRO = "pro"
    TEAM = "team"

print(f"list(Plan): {list(Plan)}")

list(Plan): [<Plan.FREE: 'free'>, <Plan.PRO: 'pro'>, <Plan.TEAM: 'team'>]


## 8. Despacho com `match/case`

`match` com membros de Enum deixa o fluxo explícito. Sempre inclua `case _:` (ou, no avançado, `assert_never` para o type checker
garantir que todos os casos foram tratados).

In [16]:
def next_action(status: OrderStatus) -> str:
    match status:
        case OrderStatus.PENDING:
            return "cobrar cliente"
        case OrderStatus.PAID:
            return "separar e enviar"
        case OrderStatus.SHIPPED:
            return "acompanhar entrega"
        case OrderStatus.CANCELLED:
            return "estornar se necessário"
        case _:
            raise ValueError(f"status desconhecido: {status!r}")

for s in OrderStatus:
    print(f"{s.name:<10} -> {next_action(s)}")

PENDING    -> cobrar cliente
PAID       -> separar e enviar
SHIPPED    -> acompanhar entrega
CANCELLED  -> estornar se necessário


Vários membros no mesmo `case` com `|`:

In [17]:
def is_final(status: OrderStatus) -> bool:
    match status:
        case OrderStatus.SHIPPED | OrderStatus.CANCELLED:
            return True
        case _:
            return False

print(f"{{s.name: is_final(s) for s in OrderStatus}}: {({s.name: is_final(s) for s in OrderStatus})}")

{s.name: is_final(s) for s in OrderStatus}: {'PENDING': False, 'PAID': False, 'SHIPPED': True, 'CANCELLED': True}


## 9. Valores que não são números

O valor pode ser qualquer objeto: string, tupla, até um objeto mais rico. Escolha valores **estáveis e significativos**
quando eles saem do programa (JSON, banco).

In [18]:
class Color(Enum):
    RED = "R"
    GREEN = "G"
    BLUE = "B"

class HttpMethod(Enum):
    GET = ("GET", False)       # (verbo, tem corpo?)
    POST = ("POST", True)
    DELETE = ("DELETE", False)

def create_car(color: Color) -> str:
    match color:
        case Color.RED:
            return "Um carro vermelho"
        case Color.GREEN:
            return "Um carro verde"
        case Color.BLUE:
            return "Um carro azul"
        case _:
            return f"A cor {color!r} não existe aqui"

print(f'create_car(Color("B")): {create_car(Color("B"))}')
print(f"HttpMethod.POST.value: {HttpMethod.POST.value}")
print(f"tem corpo?: {HttpMethod.POST.value[1]}")

create_car(Color("B")): Um carro azul
HttpMethod.POST.value: ('POST', True)
tem corpo?: True


> Repare em `case _: return f"A cor {color!r} ..."`: use a **variável** recebida, não a classe (`{Color}` imprimiria `<enum 'Color'>`).
> Membros com vários dados ficam bem mais legíveis com `__init__`: em vez de `HttpMethod.POST.value[1]` (o que é o índice 1?),
> você acessa `HttpMethod.POST.has_body`. Reescrevendo o `HttpMethod` assim (detalhes no intermediário):

In [19]:
class HttpMethod(Enum):
    GET = ("GET", False)
    POST = ("POST", True)
    DELETE = ("DELETE", False)

    def __init__(self, verb: str, has_body: bool):
        self.verb = verb
        self.has_body = has_body

print(f"HttpMethod.POST.verb: {HttpMethod.POST.verb}")
print(f"HttpMethod.POST.has_body: {HttpMethod.POST.has_body}")
print(f"[m.verb for m in HttpMethod if not m.has_body]: {[m.verb for m in HttpMethod if not m.has_body]}")

HttpMethod.POST.verb: POST
HttpMethod.POST.has_body: True
[m.verb for m in HttpMethod if not m.has_body]: ['GET', 'DELETE']


## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | `Enum` puro não é igual ao valor bruto (`Status.PAID == "paid"` é `False`) | §5 | comparar membros, converter a entrada com `Status(valor)` |
| 2 | `auto()` muda valores quando alguém insere um membro no meio | §6 | valores explícitos (ou `StrEnum` + `auto()`) se forem persistidos |
| 3 | valor repetido vira alias em silêncio | §7 | `@unique` |

### Novas

**1. Membro com valor `0` é verdadeiro no `Enum`… e falso no `IntEnum`.**

In [20]:
from enum import IntEnum


class Retries(Enum):
    NONE = 0
    SOME = 3


class IntRetries(IntEnum):
    NONE = 0
    SOME = 3


print(f"bool(Retries.NONE): {bool(Retries.NONE)}")
print(f"bool(IntRetries.NONE): {bool(IntRetries.NONE)}")

bool(Retries.NONE): True
bool(IntRetries.NONE): False


Um `if config.retries:` muda de comportamento quando alguém troca `Enum` por `IntEnum` "para facilitar". Compare explicitamente
(`if config.retries is not Retries.NONE`) em vez de depender da verdade do membro.

**2. `case RED:` sem o prefixo do Enum **captura** em vez de comparar.**

In [21]:
capture_demo = """
def classify(color):
    match color:
        case RED:            # nome solto = padrão de captura: casa com QUALQUER valor
            return f"vermelho? recebi {RED!r}"
"""
capture_ns: dict = {}
exec(capture_demo, capture_ns)
print(f"classify('azul'): {capture_ns['classify']('azul')}")

classify('azul'): vermelho? recebi 'azul'


Em `match`, um nome solto é uma **variável** que recebe o valor; só nomes com ponto (`Color.RED`) comparam. Se houver outro `case`
depois, o Python acusa `SyntaxError` ("makes remaining patterns unreachable"); se for o último, passa calado e casa com tudo.
Sempre `case Color.RED:`.

**3. `StrEnum` e `str` colidem como chave de dicionário.**

In [22]:
from enum import StrEnum


class Key(StrEnum):
    OK = "ok"


class PlainKey(Enum):
    OK = "ok"


mixed = {"ok": "valor antigo (str)", Key.OK: "valor novo (membro)"}
print(f"mixed: {mixed}")
print(f"len(mixed): {len(mixed)}")
print(f"{{PlainKey.OK: 1}}.get('ok'): {({PlainKey.OK: 1}).get('ok')}")

mixed: {'ok': 'valor novo (membro)'}
len(mixed): 1
{PlainKey.OK: 1}.get('ok'): None


`StrEnum` herda `__hash__` e `__eq__` de `str`: `Key.OK` e `"ok"` são **a mesma chave** — a segunda atribuição sobrescreve a
primeira. Já com `Enum` puro, procurar pelo valor bruto não acha nada. Escolha um tipo de chave e converta na entrada.

## 10. Resumo

| conceito | lembrete |
|---|---|
| membro | singleton com `.name` e `.value` |
| entrada | `Enum(valor)` para dados externos; `Enum[nome]` para config/CLI |
| comparação | `is` (idiomático) ou `==` entre membros; `Enum` puro ≠ valor bruto |
| iteração | `list(E)`, `len(E)`, `valor in E` (3.12+) |
| `auto()` | gera valores; cuidado se forem persistidos |
| aliases | mesmo valor = mesmo membro; `@unique` proíbe |
| tabelas | Enum é hashable → `dict[Enum, ...]` substitui `if/elif` |
| `match/case` | despacho explícito; sempre trate o `_` |

### Exercícios
1. Crie `PaymentMethod` (`pix`, `card`, `boleto`) e uma tabela `dict[PaymentMethod, float]` com a taxa de cada um.
2. Escreva `parse_priority(raw: str)` que aceite o **nome** (`"HIGH"`) e devolva o membro, com erro listando as opções.
3. Crie um Enum com um alias intencional (`COLOUR`/`COLOR`) e verifique o que aparece em `list()` e em `__members__`.

➡️ Próximo: [`02_enum_intermediario.ipynb`](02_enum_intermediario.ipynb)